# Simplified CMB TT Spectrum With RECFAST

This notebook downloads the simplified CMB code from GitHub, compiles the C++ solver, and computes the Sachs-Wolfe, Doppler, ISW, and full TT spectra from `ell = 2` to `ell = 2500`.

The parameter vector is

```text
theta = [log10(10^9 A_s), omega_cdm, omega_b, h, n_s]
```

where `omega_cdm = Omega_cdm h^2` and `omega_b = Omega_b h^2`.

In [ ]:
# Download or update the repository.
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/tsmith2/GGI_cosmology_notebooks.git"
REPO_DIR = Path("/content/GGI_cosmology_notebooks")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

# Find the package even if it lives inside a course subdirectory.
matches = list(REPO_DIR.rglob("simplified_CMB/cpp/two_fluid_tt.cpp"))
if not matches:
    matches = [p for p in REPO_DIR.rglob("two_fluid_tt.cpp") if "simplified_CMB" in str(p)]
if not matches:
    raise FileNotFoundError("Could not find simplified_CMB in the cloned repository.")

PACKAGE_DIR = matches[0].parents[1]
CPP_DIR = PACKAGE_DIR / "cpp"
os.chdir(PACKAGE_DIR)
print("Working in", PACKAGE_DIR)

In [ ]:
# Compile the C++ solver for the current Colab/Linux runtime.
# The clean rebuild avoids accidentally using a binary built on another machine.
subprocess.run(["make", "-C", str(CPP_DIR), "rebuild"], check=True)

In [ ]:
# Import the teaching wrapper.
import sys
import time
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(PACKAGE_DIR / "python"))

from two_fluid_recfast_theta_colab import (
    FID_THETA,
    plot_source_spectra,
    source_spectra,
    tt_spectrum,
)

plt.rcParams.update({"font.size": 12})

## Fiducial Cosmology

The next cell sets the cosmological parameters explicitly.  These values are close to a Planck-like LCDM cosmology.

In [ ]:
# theta = [log10(10^9 A_s), omega_cdm, omega_b, h, n_s]
theta = np.array([np.log10(2.1), 0.1201, 0.0223, 0.67, 0.965], dtype=float)

print("theta =", theta)
print("FID_THETA in the wrapper is", FID_THETA)

In [ ]:
# Compute all four spectra.  The ISW calculation is the slowest part.
t0 = time.perf_counter()
spectra = source_spectra(theta)
dt = time.perf_counter() - t0
print(f"Computed full, SW, Doppler, and ISW spectra in {dt:.2f} s")

for mode, spec in spectra.items():
    print(f"{mode:7s}: ell = {spec.ell[0]}..{spec.ell[-1]}, N = {spec.ell.size}")

In [ ]:
# Plot D_ell = ell(ell+1) C_ell / 2pi in microkelvin squared.
fig, ax = plot_source_spectra(spectra, xscale="log")
ax.set_title("Two-fluid TT source decomposition")
plt.show()

In [ ]:
# Example: compute just the full spectrum for a different cosmology.
theta_alt = theta.copy()
theta_alt[3] = 0.70  # change h only

t0 = time.perf_counter()
spec_alt = tt_spectrum(theta_alt, source_mode="full")
print(f"Computed alternate full spectrum in {time.perf_counter() - t0:.2f} s")

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(spectra["full"].ell, spectra["full"].dell_over_2pi, label="fiducial", lw=2)
ax.plot(spec_alt.ell, spec_alt.dell_over_2pi, label="h = 0.70", lw=2)
ax.set_xscale("log")
ax.set_xlim(2, 2500)
ax.set_xlabel(r"Multipole $\ell$")
ax.set_ylabel(r"$\ell(\ell+1)C_\ell^{TT}/2\pi\;[\mu{\rm K}^2]$")
ax.grid(alpha=0.25)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()